In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import cross_val_score

In [3]:
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
dataset = np.loadtxt('data/housing.csv')
dataset.shape

(506, 14)

In [8]:
X_full, y_full = dataset[:, 0:13], dataset[:, 13]
n_samples  = X_full.shape[0]
n_features = X_full.shape[1] 

#### 1. Creat missing values in the dataset

In [12]:
# create missing data
rng = np.random.RandomState(0)
missing_rate = 0.5
n_missing_samples = int(np.floor(n_samples * n_features * missing_rate))

missing_features = rng.randint(0, n_features, n_missing_samples)
# lo, hi, numerber of samples in [lo, hi]
missing_samples = rng.randint(0, n_samples, n_missing_samples)

In [13]:
X_missing = X_full.copy()
y_missing = y_full.copy()

X_missing[missing_samples, missing_features] = np.nan
X_missing = pd.DataFrame(X_missing)

#### 2. Use 0 and mean to impute missing values

In [14]:
from sklearn.impute import SimpleImputer
imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
X_missing_mean = imp_mean.fit_transform(X_missing)

imp_0 = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0)
X_missing_0 = imp_0.fit_transform(X_missing)

In [15]:
pd.DataFrame(X_missing_mean).isnull().sum()

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
dtype: int64

#### 3. Use Random Forest to impute missing values

In [ ]:
X_missing_reg = X_missing.copy()
# Sort indexes of features by their numbers of missing values from smallest to largest
sortindex = np.argsort(X_missing_reg.isnull().sum(axis=0)).values

In [17]:
for i in sortindex:

    # new feature matrix: features without missing values + original target
    # new target: the features needed to be imputed
    df = X_missing_reg
    fillc = df.iloc[:,i]
    df = pd.concat([df.iloc[:, df.columns != i], pd.DataFrame(y_full)], axis=1)

    # impute 0 for new feature matrix
    df_0 = SimpleImputer(missing_values=np.nan, strategy='constant', fill_value=0).fit_transform(df)
    Ytrain = fillc[fillc.notnull()]
    Ytest  = fillc[fillc.isnull()]
    Xtrain = df_0[Ytrain.index, :]
    Xtest  = df_0[Ytest.index, :]

    # impute missing values using RF
    rfc = RandomForestRegressor(n_estimators=100)
    rfc = rfc.fit(Xtrain, Ytrain)
    Ypredict = rfc.predict(Xtest)

    # Put the imputed feature into our orginal feature matrix
    X_missing_reg.loc[X_missing_reg.iloc[:,i].isnull(), i] = Ypredict

In [18]:
X_missing_reg.isnull().sum()

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
dtype: int64

#### 4. Test different methods

In [19]:
X = [X_full, X_missing_mean, X_missing_0, X_missing_reg]
mse = []

for x in X:
    estimator = RandomForestRegressor(random_state=0, n_estimators=100)
    scores = cross_val_score(estimator, x, y_full, scoring='neg_mean_squared_error', cv=5).mean()
    mse.append(scores * (-1))

In [20]:
[*zip(["x_full", "x_missing_mean", "x_missing_0", "x_missing_reg"], mse)]

[('x_full', np.float64(21.60580241374489)),
 ('x_missing_mean', np.float64(40.82981173393515)),
 ('x_missing_0', np.float64(49.79605629114733)),
 ('x_missing_reg', np.float64(18.87026632663561))]